# Hybrid Document AI — OCR + KIE (production-grade)

**Ứng tuyển AI Engineer @ VNPAY** — track *NLP · Computer Vision · MLOps*.

Notebook này tổng hợp & cho phép **chạy lại toàn bộ project từ đầu** trên cùng máy này.

## Kiến trúc 3-layer (chuẩn Banking/Enterprise)
| Layer | Thành phần |
|---|---|
| **Processing** (multi-model) | Quality → Layout → Text-Detect → OCR (RapidOCR/PP-OCR ONNX) → **KIE 2-tier** → hybrid router → VLM fallback |
| **Serving** | **Dynamic micro-batcher** (chạy thật) · stage queues · CPU/VRAM caps · Triton + vLLM (artifact) |
| **Deployment + MLOps** | FastAPI · Docker/Compose · Prometheus+drift · **model registry/versioning** · **eval-as-CI-gate** · K8s HPA |

**KIE không chỉ regex:** Tier-1 = candidate-gen (regex/keyword/layout-graph) → feature vector → **scikit-learn classifier** (trained, versioned, eval-gated). Tier-2 = **VLM OCR-free** (Donut/Qwen) cho hard case qua confidence router.

## Artifacts đã publish
- 🤗 Model: https://huggingface.co/banhchungtuongot/hybrid-docai-kie
- 🤗 Dataset: https://huggingface.co/datasets/banhchungtuongot/hybrid-docai-receipts
- 🐙 Code: https://github.com/NguyenDucAnforwork/hybrid-document-ai

## Ràng buộc đã tôn trọng
≤2h build · ≤15GB disk (**artifacts để trên `/data` vì `/home` đầy 100%**) · ≤4GB VRAM (VLM off-box: `disabled`/`api`/`remote_gpu`).


## 0. Prerequisites & guideline
- Chạy tuần tự từ trên xuống. Repo root = thư mục chứa notebook này (`/home/nvidia-lab/nltk_data`).
- **Heavy artifacts (venv/data/models) đặt trên `/data`** — đặt `DOCAI_WORKSPACE` TRƯỚC khi import `docai` (config đọc env lúc import).
- Nếu chạy máy khác: đổi `DOCAI_WORKSPACE` sang chỗ có ≥3GB trống.


In [ ]:
import os, sys, subprocess, json
# QUAN TRỌNG: set workspace TRƯỚC khi import docai
os.environ['DOCAI_WORKSPACE'] = '/data/nvidia-ai-workspace'
REPO = os.path.dirname(os.path.abspath('test.ipynb')) or os.getcwd()
os.chdir(REPO); sys.path.insert(0, REPO)
WS = os.environ['DOCAI_WORKSPACE']
print('repo =', REPO); print('workspace =', WS)


### 0.1 Cài dependencies
Không cần `torch` cho build chạy thật (VLM ở `api` mode). ~743MB.


In [ ]:
# subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
print('Nếu chưa cài: bỏ comment dòng trên. Trên máy này đã có venv /data/nvidia-ai-workspace/venv')


## 1. Sinh dữ liệu synthetic (reproducible, seed=42)
Receipt có gold label (merchant/date/total/invoice/payment), không dữ liệu cá nhân. SROIE là target thật (xem `scripts/`), synthetic giữ demo chạy nhanh + có gold chính xác.


In [ ]:
from docai.synth import generate
recs = generate(f'{WS}/data/receipts', n=120, seed=42)
print('generated', len(recs), 'receipts ->', f'{WS}/data/receipts')
print('1 sample gold:', recs[0]['gold'])


## 2. Training pipeline — KIE field classifier (MLOps)
data → features (OCR-conf, layout position, keyword proximity, is_max_number, is_largest_font) → fit sklearn → eval → **đăng ký vào registry** (versioned, v1→v2).
Cell train dưới dùng `--version v2` (active). Registry giữ cả v1 để truy vết.


In [ ]:
r = subprocess.run([sys.executable,'training/train_kie.py','--data',f'{WS}/data/receipts',
                     '--version','v2','--seed','42'], capture_output=True, text=True)
print(r.stdout[-900:]); print(r.stderr[-300:] if r.returncode else 'OK')


In [ ]:
# Model registry (versioning + metrics, để audit/traceability)
print(open(f'{WS}/models/registry.yaml').read())


## 3. Benchmark end-to-end + eval-as-CI-gate
Chạy **OCR thật** trên ảnh render → KIE → so gold. Exit≠0 nếu macro-F1 < threshold (cổng CI).


In [ ]:
r = subprocess.run([sys.executable,'scripts/run_benchmark.py','--data',f'{WS}/data/receipts',
                     '--out','docs/logs','--f1-threshold','0.6'], capture_output=True, text=True)
print(r.stdout[-1200:]); print('GATE', 'PASS' if r.returncode==0 else 'FAIL')


## 4. Inference qua API (FastAPI TestClient — không cần chạy server riêng)


In [ ]:
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)   # kích hoạt startup: warm-up OCR + load KIE version
print('health:', client.get('/health').json())


In [ ]:
# 4.1 Single document
img = f'{WS}/data/receipts/images/rcpt_0000.png'
res = client.post('/documents/extract', files={'file': open(img,'rb')}).json()
print(json.dumps(res, indent=2, ensure_ascii=False))


In [ ]:
# 4.2 Batch job (async orchestration, per-doc state machine, retry, dead-letter)
files = [('files', open(f'{WS}/data/receipts/images/rcpt_{i:04d}.png','rb')) for i in range(5)]
job = client.post('/batch_jobs', files=files).json()
print('job:', json.dumps(job, indent=2))
print('results[0]:', json.dumps(client.get(f"/batch_jobs/{job['job_id']}/results").json()['results'][0], indent=2, ensure_ascii=False))


In [ ]:
# 4.3 Prometheus metrics + drift signals
m = client.get('/metrics').text
print('\n'.join(l for l in m.splitlines() if l.startswith(('documents_processed','vlm_fallback','human_review','field_confidence_count','input_blur_score_count'))))


## 5. (Optional) Tải model/data đã publish từ HuggingFace


In [ ]:
# from huggingface_hub import hf_hub_download, snapshot_download
# hf_hub_download('banhchungtuongot/hybrid-docai-kie','kie/v1/model.joblib')
# snapshot_download('banhchungtuongot/hybrid-docai-receipts', repo_type='dataset')
print('Model: banhchungtuongot/hybrid-docai-kie | Dataset: banhchungtuongot/hybrid-docai-receipts')


## 6. Production deployment (swap-by-config, không sửa code)
Local dùng `memory` queue + filesystem + OCR in-process. Production đổi qua `configs/app.yaml` + `deploy/docker-compose.yml`:
```yaml
queue:   {backend: redis}
storage: {backend: minio}
serving: {ocr_via: triton}      # ONNX drop-in serving/triton/model_repository/*
vlm:     {mode: remote_gpu, base_url: http://vllm:8000/v1}
```
```bash
cd deploy && docker compose up        # api + redis + minio + triton + vllm + prometheus
kubectl apply -f deploy/hpa.yaml      # autoscale theo queue_size
```
Serving Layer chạy thật trong build này: **dynamic micro-batcher** (`docai/batcher.py`, test `tests/test_pipeline.py::test_batcher_groups`). Triton/vLLM là artifact production.


## 7. Kết quả (đo thật, end-to-end real OCR, 40 receipts) — KIE **v2**
| field | F1 | exact-match |
|---|---|---|
| merchant_name | 1.00 | 1.00 |
| date | 1.00 | 1.00 |
| total_amount | 1.00 | 1.00 |
| payment_method | 0.98 | 0.98 |
| invoice_id | 0.92 | 0.90 |
| **macro-F1** | **0.98** | all-required-correct **1.00** |

Latency p50 **334ms** / p95 **515ms** (CPU). **v1→v2 (0.865→0.98)** nhờ debug eval:
(1) **layout-graph line-grouping** (OCR tách 'ABC'+'MART') → merchant 0.45→1.0;
(2) **money regex chặt** (bắt buộc dấu phân cách nghìn) → date/ID không bị nhận nhầm là tiền → total về 1.0.
`invoice_id` 0.90 là trần OCR thật (đọc nhầm 1↔l) → low-conf route human review.

### Tài liệu chi tiết
`PLAN.md` · `docs/lessons-learned.md` (ADR + debug v1→v2) · `docs/debug-workflows.md` · `docs/reproduce.md` · `docs/logs/`
